# Penggabungan dan Pemrosesan Dataset Blibli & Tokopedia

Tujuan notebook ini:
1. Menggabungkan `dataset_blibli_berimbang_35k.csv`, `dataset_tokopedia_final.csv`, `rafie_tokopedia_dataset.csv`, dan `rafie2_tokopedia_dataset.csv`
2. Mengecek total baris awal
3. Memfilter rating 1, 2, dan 3 agar minimal memiliki review dengan $\ge$ 3 kata (menghindari review sangat singkat seperti 'Good', 'Ok', 'O')
4. Memfilter sisa data (rating 4 & 5) agar hanya memiliki review dengan $\ge$ 8 kata
5. Mengambil sampel rating 4 & 5 hingga total keseluruhan dataset menjadi tepat 40.000

In [5]:
import pandas as pd
import numpy as np

In [6]:
# 1. Load datasets
df_blibli = pd.read_csv('data/dataset_blibli_berimbang_35k.csv')
df_tokped = pd.read_csv('data/dataset_tokopedia_final.csv')
df_rafie = pd.read_csv('data/rafie_tokopedia_dataset.csv')
df_rafie2 = pd.read_csv('data/rafie2_tokopedia_dataset.csv')

# Tambahkan kolom Platform ke Blibli agar seragam
if 'Platform' not in df_blibli.columns:
    df_blibli['Platform'] = 'Blibli'

# Map columns untuk dataset rafie agar seragam
df_rafie = df_rafie.rename(columns={'Customer Rating': 'Rating', 'Customer Review': 'Review'})
df_rafie['Platform'] = 'Tokopedia'

df_rafie2 = df_rafie2.rename(columns={'Customer Rating': 'Rating', 'Customer Review': 'Review'})
df_rafie2['Platform'] = 'Tokopedia'

# Ambil kolom yang relevan (Platform, Rating, Review)
cols = ['Platform', 'Rating', 'Review']
df_blibli = df_blibli[[c for c in cols if c in df_blibli.columns]]
df_tokped = df_tokped[[c for c in cols if c in df_tokped.columns]]
df_rafie = df_rafie[cols]
df_rafie2 = df_rafie2[cols]

# 2. Gabungkan semua dataset
df_merged = pd.concat([df_blibli, df_tokped, df_rafie, df_rafie2], ignore_index=True)
df_merged = df_merged.dropna(subset=['Review']) # Hapus yang review-nya kosong
df_merged['Review'] = df_merged['Review'].astype(str)

# Cek Total Awal
print(f"Total data gabungan awal: {len(df_merged)} baris")
print(df_merged['Rating'].value_counts())

Total data gabungan awal: 114177 baris
Rating
5    93838
1     7181
3     7026
4     3159
2     2973
Name: count, dtype: int64


In [7]:
# Fungsi untuk menghitung kata
def count_words(text):
    return len(str(text).split())

# 3. Ambil data dengan rating 1, 2, dan 3 (filter minimal 3 kata)
df_123_raw = df_merged[df_merged['Rating'].isin([1, 2, 3])]
df_123 = df_123_raw[df_123_raw['Review'].apply(count_words) >= 3]
print(f"Total data rating 1, 2, 3 setelah filter (>= 3 kata): {len(df_123)} baris (sebelumnya {len(df_123_raw)})")

# 4. Ambil data rating 4 dan 5, lalu filter yang jumlah katanya >= 8
df_45 = df_merged[df_merged['Rating'].isin([4, 5])]
df_45_filtered = df_45[df_45['Review'].apply(count_words) >= 8]
print(f"Total data rating 4 & 5 (>= 8 kata): {len(df_45_filtered)} baris")

# 5. Hitung sisa kuota untuk mencapai total 30.000
kuota_sisa = 40000 - len(df_123)
print(f"Sisa kuota untuk mencapai 30k (diambil dari rating 4&5): {kuota_sisa} baris")

# 6. Ambil sampel acak dari data rating 4 & 5 yang sudah difilter
if len(df_45_filtered) >= kuota_sisa:
    df_45_sampled = df_45_filtered.sample(n=kuota_sisa, random_state=42)
else:
    print("PERINGATAN: Data rating 4&5 setelah filter kurang dari sisa kuota yang dibutuhkan!")
    df_45_sampled = df_45_filtered

# 7. Gabungkan menjadi dataset final 30k
df_final = pd.concat([df_123, df_45_sampled], ignore_index=True)

# Acak urutan datanya agar tidak mengelompok
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTotal data akhir: {len(df_final)} baris")
print("\nDistribusi Rating Akhir:")
print(df_final['Rating'].value_counts().sort_index())

Total data rating 1, 2, 3 setelah filter (>= 3 kata): 14765 baris (sebelumnya 17180)
Total data rating 4 & 5 (>= 8 kata): 92584 baris
Sisa kuota untuk mencapai 30k (diambil dari rating 4&5): 25235 baris

Total data akhir: 40000 baris

Distribusi Rating Akhir:
Rating
1     6155
2     2769
3     5841
4      856
5    24379
Name: count, dtype: int64


In [8]:
# Simpan dataset final
output_path = 'data/dataset_gabungan_30k.csv'
df_final.to_csv(output_path, index=False)
print(f"Dataset final berhasil disimpan ke '{output_path}'")

Dataset final berhasil disimpan ke 'data/dataset_gabungan_30k.csv'


In [9]:
!pip install emoji

In [10]:
import pandas as pd
import re
import emoji
import os
from tqdm import tqdm

# 1. LOAD DATA
INPUT_FILE = "data/dataset_gabungan_30k.csv"
OUTPUT_FILE = "data/data_primer.csv"

print("Loading dataset...")
df = pd.read_csv(INPUT_FILE)

df['Review'] = df['Review'].fillna("").astype(str)

kamus_alay = {
    "bgt": "banget", "jg": "juga", "gbs": "tidak bisa", "tp": "tapi",
    "udh": "sudah", "sdh": "sudah", "blm": "belum", "krn": "karena",
    "bkn": "bukan", "dpt": "dapat", "ga": "tidak", "gak": "tidak",
    "ndak": "tidak", "utk": "untuk", "dr": "dari", "aja": "saja",
    "kmrn": "kemarin", "pake": "pakai", "pke": "pakai", "engga": "tidak",
    "Original": "asli", "ori": "asli", "mantap": "bagus", "mntp": "bagus",
    "klo": "kalau", "kl": "kalau", "bs": "bisa", "sampe": "sampai"
}

def normalisasi_kata(text):
    words = text.split()
    words_clean = [kamus_alay.get(w, w) for w in words]
    return " ".join(words_clean)


def clean_text_for_bert(text: str) -> str:
    # 1. Hapus URL, mention, hashtag terlebih dahulu
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\S+|#\S+", " ", text)
    
    # 2. Bersihkan karakter selain huruf/angka, tapi JANGAN HAPUS emoji dulu
    # Kita tambahkan flag khusus agar simbol emoji tidak ikut terhapus di tahap ini
    text = re.sub(r"[^a-zA-Z0-9\s.,!?\U00010000-\U0010ffff]", " ", text)
    
    # 3. Normalisasi kata alay
    text = normalisasi_kata(text)
    
    # 4. Baru ubah emoji menjadi teks kode di tahap paling akhir
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 5. Rapikan spasi ganda dan hilangkan tanda petik dua bawaan demojize jika ada
    text = text.replace(":", " ")
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

tqdm.pandas()  


df['Review_Clean'] = df['Review'].progress_apply(clean_text_for_bert)

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nTotal data setelah difilter: {len(df)} baris.")
print(f"File bersih disimpan di: {OUTPUT_FILE}")

# Preview data hasil cleaning
print("\nPreview Hasil (Top 3):")
display(df[['Review', 'Review_Clean']].head(3))

Loading dataset...


100%|██████████| 40000/40000 [00:10<00:00, 3821.99it/s]



Total data setelah difilter: 40000 baris.
File bersih disimpan di: data/data_primer.csv

Preview Hasil (Top 3):


,Review,Review_Clean
0,perdana beli oli MOTUL belum di coba semoga be...,perdana beli oli MOTUL belum di coba semoga be...
1,Toko langganan Kantor. Harga tdk terlalu mahal...,Toko langganan Kantor. Harga tdk terlalu mahal...
2,"Seller fast respon, kualitasnya bagus juga unt...","Seller fast respon, kualitasnya bagus juga unt..."
